# Data Cleaning and Preparation

This notebook prepares the AeroFlow datasets for analysis following the data-quality checks completed during profiling.

The cleaning process focuses on correcting data types, standardising fields where required and creating analysis-ready datasets while preserving the original raw files.

## 1. Import Libraries and Load Data

In [3]:
import pandas as pd

In [10]:
parts = pd.read_csv("../data/raw/parts_master.csv")
purchase_orders = pd.read_csv("../data/raw/purchase_orders.csv")
quality_incidents = pd.read_csv("../data/raw/quality_incidents.csv")
supply_chain_history = pd.read_csv("../data/raw/supply_chain_history.csv")

## 2. Convert Date Columns

In [11]:
purchase_orders["order_date"] = pd.to_datetime(purchase_orders["order_date"])
purchase_orders["promised_date"] = pd.to_datetime(purchase_orders["promised_date"])
purchase_orders["receipt_date"] = pd.to_datetime(purchase_orders["receipt_date"])

quality_incidents["incident_date"] = pd.to_datetime(quality_incidents["incident_date"])

supply_chain_history["date"] = pd.to_datetime(supply_chain_history["date"])

In [12]:
purchase_orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 29666 entries, 0 to 29665
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   po_id          29666 non-null  str           
 1   supplier_id    29666 non-null  str           
 2   site_id        29666 non-null  str           
 3   part_id        29666 non-null  str           
 4   order_date     29666 non-null  datetime64[us]
 5   promised_date  29666 non-null  datetime64[us]
 6   receipt_date   29666 non-null  datetime64[us]
 7   ordered_qty    29666 non-null  int64         
 8   received_qty   29666 non-null  int64         
dtypes: datetime64[us](3), int64(2), str(4)
memory usage: 2.0 MB


In [13]:
quality_incidents.info()

<class 'pandas.DataFrame'>
RangeIndex: 368 entries, 0 to 367
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   incident_id      368 non-null    str           
 1   incident_date    368 non-null    datetime64[us]
 2   part_id          368 non-null    str           
 3   supplier_id      368 non-null    str           
 4   site_id          368 non-null    str           
 5   defect_severity  368 non-null    str           
 6   defect_type      368 non-null    str           
 7   scrap_qty        368 non-null    int64         
dtypes: datetime64[us](1), int64(1), str(6)
memory usage: 23.1 KB


In [14]:
supply_chain_history.info()

<class 'pandas.DataFrame'>
RangeIndex: 280800 entries, 0 to 280799
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   date                 280800 non-null  datetime64[us]
 1   site_id              280800 non-null  str           
 2   part_id              280800 non-null  str           
 3   planned_maintenance  280800 non-null  bool          
 4   consumption_qty      280800 non-null  int64         
 5   on_hand_qty          280800 non-null  int64         
 6   backorder_qty        280800 non-null  int64         
 7   blocked_qty          280800 non-null  int64         
 8   forecast_qty         280800 non-null  int64         
 9   forecast_type        280800 non-null  str           
 10  forecast_uplift_pct  280800 non-null  float64       
dtypes: bool(1), datetime64[us](1), float64(1), int64(5), str(3)
memory usage: 21.7 MB


In [15]:
parts["shelf_life_days"] = parts["shelf_life_days"].astype("Int64")

In [16]:
parts.info()

<class 'pandas.DataFrame'>
RangeIndex: 300 entries, 0 to 299
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   part_id              300 non-null    str    
 1   part_family          300 non-null    str    
 2   criticality_class    300 non-null    str    
 3   unit_cost            300 non-null    float64
 4   lead_time_days       300 non-null    int64  
 5   supplier_id_primary  300 non-null    str    
 6   supplier_risk_class  300 non-null    str    
 7   is_repairable        300 non-null    str    
 8   shelf_life_days      26 non-null     Int64  
dtypes: Int64(1), float64(1), int64(1), str(6)
memory usage: 21.5 KB


## 3. Missing Value Treatment

The missing values in `shelf_life_days` were retained because profiling suggested the field does not apply to every part. The column was converted to a nullable integer datatype so whole-day values could be stored while preserving missing records.

In [ ]:
## 4. Create Analysis Fields

Additional fields were created to support delivery and supplier performance analysis while keeping the original source columns unchanged.

In [17]:
purchase_orders["delivery_variance_days"] = (
    purchase_orders["receipt_date"] - purchase_orders["promised_date"]
).dt.days

In [ ]:
purchase_orders[["promised_date", "receipt_date", "delivery_variance_days"]].head()

,promised_date,receipt_date,delivery_variance_days
0,2022-05-28,2022-05-29,1
1,2022-07-31,2022-08-02,2
2,2022-11-12,2022-11-16,4
3,2023-03-07,2023-03-09,2
4,2023-04-28,2023-04-29,1


In [19]:
purchase_orders["on_time_flag"] = (
    purchase_orders["receipt_date"] <= purchase_orders["promised_date"]
)

In [20]:
purchase_orders["in_full_flag"] = (
    purchase_orders["received_qty"] == purchase_orders["ordered_qty"]
)

In [21]:
purchase_orders["otif_flag"] = (
    purchase_orders["on_time_flag"] & purchase_orders["in_full_flag"]
)

In [ ]:
purchase_orders[["on_time_flag", "in_full_flag", "otif_flag"]].value_counts()

on_time_flag  in_full_flag  otif_flag
False         True          False        14522
True          True          True         11789
False         False         False         2046
True          False         False         1309
Name: count, dtype: int64

In [ ]:
### Delivery Performance Fields

Three Boolean fields were created to support delivery performance analysis:

- `on_time_flag` identifies orders received on or before the promised date.
- `in_full_flag` identifies orders where the received quantity matches the ordered quantity.
- `otif_flag` identifies orders meeting both conditions.

The resulting OTIF count was 11,789 orders, representing 39.74% of purchase orders.

In [23]:
purchase_orders["actual_lead_time_days"] = (
    purchase_orders["receipt_date"] - purchase_orders["order_date"]
).dt.days

In [24]:
purchase_orders["actual_lead_time_days"].describe()

count    29666.000000
mean        42.971516
std         15.862033
min          8.000000
25%         32.000000
50%         41.000000
75%         52.000000
max        131.000000
Name: actual_lead_time_days, dtype: float64

An `actual_lead_time_days` field was calculated from the difference between the order and receipt dates. This provides the observed lead time for each purchase order and can later be compared with expected part lead times during analysis.

## 5. Prepare Supply Chain History

The supply chain history data required no corrective cleaning beyond date conversion. Additional fields were created only where they support later inventory and supply-risk analysis.

In [25]:
supply_chain_history["stockout_flag"] = (
  supply_chain_history["on_hand_qty"] == 0
)

In [26]:
supply_chain_history["stockout_flag"].value_counts()

stockout_flag
False    279186
True       1614
Name: count, dtype: int64

In [27]:
stockout_pct = (
    supply_chain_history["stockout_flag"].sum()
    / len(supply_chain_history)
    * 100
)

round(stockout_pct, 2)

np.float64(0.57)

### Stockout Flag

A `stockout_flag` was created to identify weekly part-site records where `on_hand_qty` was zero.

1,614 records (0.57%) were identified as stockouts. These records were retained as valid operational events for further analysis.

In [28]:
supply_chain_history["backorder_flag"] = (
    supply_chain_history["backorder_qty"] > 0
)

In [29]:
supply_chain_history["backorder_flag"].value_counts()

backorder_flag
False    278466
True       2334
Name: count, dtype: int64

### Backorder Flag

A `backorder_flag` was created to identify weekly part-site records where `backorder_qty` was greater than zero.

2,334 records (0.83%) were identified as having a backorder.

## 6. Final Validation

Final validation checks were completed before exporting the prepared datasets to confirm that the cleaning process preserved the expected records and data quality.

In [30]:
print("Parts:", parts.shape)
print("Purchase Orders:", purchase_orders.shape)
print("Quality Incidents:", quality_incidents.shape)
print("Supply Chain History:", supply_chain_history.shape)

Parts: (300, 9)
Purchase Orders: (29666, 14)
Quality Incidents: (368, 8)
Supply Chain History: (280800, 13)


In [31]:
parts.isnull().sum()

part_id                  0
part_family              0
criticality_class        0
unit_cost                0
lead_time_days           0
supplier_id_primary      0
supplier_risk_class      0
is_repairable            0
shelf_life_days        274
dtype: int64

In [32]:
purchase_orders.isnull().sum()

po_id                     0
supplier_id               0
site_id                   0
part_id                   0
order_date                0
promised_date             0
receipt_date              0
ordered_qty               0
received_qty              0
delivery_variance_days    0
on_time_flag              0
in_full_flag              0
otif_flag                 0
actual_lead_time_days     0
dtype: int64

In [33]:
quality_incidents.isnull().sum()

incident_id        0
incident_date      0
part_id            0
supplier_id        0
site_id            0
defect_severity    0
defect_type        0
scrap_qty          0
dtype: int64

In [34]:
supply_chain_history.isnull().sum()

date                   0
site_id                0
part_id                0
planned_maintenance    0
consumption_qty        0
on_hand_qty            0
backorder_qty          0
blocked_qty            0
forecast_qty           0
forecast_type          0
forecast_uplift_pct    0
stockout_flag          0
backorder_flag         0
dtype: int64

### Duplicate Validation

In [35]:
print("Parts duplicate IDs:", parts["part_id"].duplicated().sum())

print("Purchase Order duplicate IDs:", purchase_orders["po_id"].duplicated().sum())

print(
    "Quality Incident duplicate IDs:",
    quality_incidents["incident_id"].duplicated().sum(),
)

print(
    "Supply Chain History duplicate grain:",
    supply_chain_history.duplicated(subset=["date", "site_id", "part_id"]).sum(),
)

Parts duplicate IDs: 0
Purchase Order duplicate IDs: 0
Quality Incident duplicate IDs: 0
Supply Chain History duplicate grain: 0


## 7. Export Prepared Data

Following final validation, the prepared datasets were exported to the processed data folder. The original raw files were left unchanged to preserve the source data.

In [36]:
parts.to_csv("../data/clean/parts_master_clean.csv", index=False)

purchase_orders.to_csv(
    "../data/clean/purchase_orders_clean.csv",
    index=False
)

quality_incidents.to_csv(
    "../data/clean/quality_incidents_clean.csv",
    index=False
)

supply_chain_history.to_csv(
    "../data/clean/supply_chain_history_clean.csv",
    index=False
)

## 8. Cleaning Summary

The four source datasets were prepared for analysis while preserving the original raw data.

Key preparation steps included:

- Converted date fields to datetime format across the relevant datasets.
- Retained missing `shelf_life_days` values where no evidence supported replacing them and converted the field to a nullable integer type.
- Created delivery performance fields for purchase orders, including delivery variance, actual lead time, on-time, in-full and OTIF flags.
- Created stockout and backorder flags to support inventory and supply-risk analysis.
- Retained valid operational values, including negative forecast adjustments, after confirming their consistency with forecast type.
- Confirmed that row counts were preserved and no duplicate business keys were introduced.
- Exported the prepared datasets to the `data/clean` folder for analysis.

The cleaned datasets are now ready for exploratory data analysis.